# 00 — Model & Method Selection
stage 0, before `01_ocr_pipeline.ipynb`, only exploratory results. All inputs live under `ocr_choices/`

## Setup
Imports, Drive auth, output folder.


In [ ]:
import csv
from pathlib import Path
from Levenshtein import distance as levenshtein_distance
import pandas as pd
from config import DRIVE_ROOT_FOLDER, DRIVE_OCR_CHOICES_FOLDER, GROUND_TRUTH_DIR, OCR_CHOICES_DIR
from drive_utils import get_drive_service, find_folder, get_or_create_folder, save_or_upload_csv

OCR_CHOICES = Path(OCR_CHOICES_DIR)
CHOICE1_DIR = OCR_CHOICES / "choice1_ocr_engine"
CHOICE2_DIR = OCR_CHOICES / "choice2_architecture"
CHOICE3_DIR = OCR_CHOICES / "choice3_diversity_validation"
EXPLORATION_OUTPUTS = CHOICE3_DIR / "exploration_outputs"


service = get_drive_service()
_root_id = find_folder(service, DRIVE_ROOT_FOLDER)
_ocr_choices_folder_id = get_or_create_folder(service, DRIVE_OCR_CHOICES_FOLDER, _root_id)
choice1_folder_id = get_or_create_folder(service, CHOICE1_DIR.name, _ocr_choices_folder_id)
choice2_folder_id = get_or_create_folder(service, CHOICE2_DIR.name, _ocr_choices_folder_id)
choice3_folder_id = get_or_create_folder(service, CHOICE3_DIR.name, _ocr_choices_folder_id)

/Users/charlottegarcia/global_venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.2.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


## Choice 1 — OCR method: Tesseract vs. GPT-4o vs. Gemini (pages 86 and 97 from the pdf)

In [2]:
# Strips brackets/quotes, collapses whitespace. No lowercasing.
def normalize_ocr(text):
    text = text.replace('[', '').replace(']', '').replace('"', '')
    return " ".join(text.split())

# Levenshtein-based character similarity, as a percentage.
def char_accuracy(a, b):
    dist = levenshtein_distance(a, b)
    return (1 - dist / max(len(a), len(b))) * 100

# Levenshtein-based word similarity, as a percentage.
def word_accuracy(a, b):
    wa, wb = " ".join(a.split()), " ".join(b.split())
    dist = levenshtein_distance(wa, wb)
    return (1 - dist / max(len(a.split()), len(b.split()))) * 100

# Fraction of lines that match exactly, position by position.
def line_accuracy(a, b):
    la = [l.strip() for l in a.splitlines() if l.strip()]
    lb = [l.strip() for l in b.splitlines() if l.strip()]
    correct = sum(1 for x, y in zip(la, lb) if x == y)
    return correct / max(len(la), len(lb)) * 100

In [3]:
#method comparison for OCR for pages 49 and 60 (actually pages 86 and 97 from the pdf)

pages = {
    49: dict(gt="1947_page49_ground_truth.txt", tesseract="page_49_comp.txt",
             gemini="1947_gc_page_49_gemini_straight_reading.txt", gpt="1947_gc_page_49_gpt_straight_reading.txt"),
    60: dict(gt="1947_page60_groundtruth.txt", tesseract="page_60_comp.txt",
             gemini="1947_gc_page_60_gemini_straight_reading.txt", gpt="1947_gc_page_60_gpt_straight_reading.txt"),
}

rows = []
for page, files in pages.items():
    gt = normalize_ocr((CHOICE1_DIR / files["gt"]).read_text(encoding="utf-8"))
    gt_raw = (CHOICE1_DIR / files["gt"]).read_text(encoding="utf-8")
    for method in ("tesseract", "gemini", "gpt"):
        raw = (CHOICE1_DIR / files[method]).read_text(encoding="utf-8")
        cand = normalize_ocr(raw)
        rows.append(dict(
            page=page, method=method,
            char_acc=round(char_accuracy(cand, gt), 2),
            word_acc=round(word_accuracy(cand, gt), 2),
            line_acc=round(line_accuracy(raw, gt_raw), 2),
        ))

#output + save results
choice1_ocr_method = pd.DataFrame(rows)
save_or_upload_csv(choice1_ocr_method, CHOICE1_DIR / "choice1_ocr_method_comparison.csv",
                    service, choice1_folder_id)
choice1_ocr_method

,page,method,char_acc,word_acc,line_acc
0,49,tesseract,93.94,59.59,0.00
1,49,gemini,94.72,63.60,0.00
2,49,gpt,98.51,90.06,0.00
3,60,tesseract,95.02,63.08,2.15
4,60,gemini,96.44,73.54,0.00
5,60,gpt,95.49,66.50,0.00


## Choice 2 — Extraction strategy: read-then-extract (S1) vs. direct (S2) vs. OCR-then-extract (S3)
Tested on PDF page 110 (page 70 in the almanac)

In [4]:
# Shared scoring helpers
def load_csv_file(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            clean_row = {}
            for k, v in row.items():
                if k is None:
                    continue
                clean_row[k] = (v if v is not None else "").strip() if isinstance(v, str) else str(v or "")
            rows.append(clean_row)
    return rows

# Lowercases and strips punctuation, for exact-match scoring.
def normalize(text):
    if text is None:
        return ""
    text = str(text).replace('[', '').replace(']', '').replace('"', '')
    return " ".join(text.split()).lower()

# Applies normalize() to every value in a list of row dicts.
def normalize_rows(rows):
    return [{k: normalize(v) for k, v in r.items()} for r in rows]

# Exact-match precision/recall/F1 and per-field accuracy against ground truth.
def semantic_accuracy(gt_rows, pred_rows):
    if not gt_rows or not pred_rows:
        return 0, 0, 0, {f: 0 for f in ["Name", "Address", "Profession", "Additional Info"]}
    fields = ["Name", "Address", "Profession", "Additional Info"]
    field_correct = {f: 0 for f in fields}
    field_total = {f: 0 for f in fields}

    def row_score(r1, r2):
        return sum(1 for k in fields if r1.get(k, "") == r2.get(k, "")) / len(fields)

    matched = 0
    for gt in gt_rows:
        best_match, best_score = None, 0
        for pred in pred_rows:
            score = row_score(gt, pred)
            if score > best_score:
                best_score, best_match = score, pred
        if best_score > 0.5:
            matched += 1
            for f in fields:
                field_total[f] += 1
                if gt.get(f, "") == best_match.get(f, ""):
                    field_correct[f] += 1

    precision = matched / len(pred_rows) if pred_rows else 0
    recall = matched / len(gt_rows) if gt_rows else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    field_acc = {f: (field_correct[f] / field_total[f] * 100 if field_total[f] > 0 else 0) for f in fields}
    return precision * 100, recall * 100, f1 * 100, field_acc


In [ ]:
#   "llm_reading_to_semantic"  -- S1, LLM-read text -> second LLM call -> semantic CSV
#   "llm_reading"              -- S2, later re-run
#   "ocr_llm_to_semantic"      -- S3, Tesseract text -> LLM call -> semantic CSV

PAGE70_ROOT = CHOICE2_DIR / "1947_page70"
p70_gt_semantic = PAGE70_ROOT / "page70_ground_truth_semantic.csv"
gt_rows_70 = normalize_rows(load_csv_file(p70_gt_semantic))

def score_semantic(run_label, source, csv_path):
    if not csv_path.exists():
        return None
    pred_rows = normalize_rows(load_csv_file(csv_path))
    p, r, f1, facc = semantic_accuracy(gt_rows_70, pred_rows)
    return dict(source=source, Run=run_label, File=csv_path.name,
                **{"Semantic Precision": round(p, 2), "Semantic Recall": round(r, 2),
                   "Semantic F1": round(f1, 2), "Profession Acc": round(facc["Profession"], 2),
                   "Name Acc": round(facc["Name"], 2), "Address Acc": round(facc["Address"], 2)})

RUN_SOURCES = [
    ("llm_reading/run_1/raw", "llm_reading_run_1_raw",
     PAGE70_ROOT / "llm_reading/run_1/raw/1947_gc_page_70_gpt_semantic_extraction.csv"),
    ("llm_reading/run_1/cleaned", "llm_reading_run_1_cleaned",
     PAGE70_ROOT / "llm_reading/run_1/cleaned/1947_gc_page_70_gpt_semantic_extraction.csv"),
    ("llm_reading_to_semantic/run_1/raw", "llm_reading_to_semantic_run_1_raw",
     PAGE70_ROOT / "llm_reading_to_semantic/run_1/raw/straight_reading_to_semantic.csv"),
    ("llm_reading_to_semantic/run_1/cleaned", "llm_reading_to_semantic_run_1_cleaned",
     PAGE70_ROOT / "llm_reading_to_semantic/run_1/cleaned/straight_reading_to_semantic.csv"),
    ("ocr_llm_to_semantic/run_1/raw", "ocr_llm_to_semantic_run_1_raw",
     PAGE70_ROOT / "ocr_llm_to_semantic/run_1/raw/straight_reading_to_semantic.csv"),
]

_rows = []
for source, run_label, csv_path in RUN_SOURCES:
    row = score_semantic(run_label, source, csv_path)
    if row:
        _rows.append(row)

strategy_eval = pd.DataFrame(_rows)

def get_semantic_row(run_name):
    return strategy_eval.loc[strategy_eval["Run"] == run_name].iloc[0]

s1 = get_semantic_row("llm_reading_to_semantic_run_1_raw")
s2 = get_semantic_row("llm_reading_run_1_raw")
s3 = get_semantic_row("ocr_llm_to_semantic_run_1_raw")

strategy_summary = pd.DataFrame([
    dict(strategy="S1", architecture="image -> text, then text -> semantic CSV (two calls)",
         semantic_f1=s1["Semantic F1"], profession_acc=s1["Profession Acc"]),
    dict(strategy="S2", architecture="image -> semantic CSV directly (one call)",
         semantic_f1=s2["Semantic F1"], profession_acc=s2["Profession Acc"]),
    dict(strategy="S3", architecture="classical OCR -> text, then text -> semantic CSV",
         semantic_f1=s3["Semantic F1"], profession_acc=s3["Profession Acc"]),
])
save_or_upload_csv(strategy_summary, CHOICE2_DIR / "choice2_strategy_comparison.csv",
                    service, choice2_folder_id)
strategy_summary


,strategy,architecture,semantic_f1,profession_acc
0,S1,"image -> text, then text -> semantic CSV (two ...",88.62,100.0
1,S2,image -> semantic CSV directly (one call),54.69,0.0
2,S3,"classical OCR -> text, then text -> semantic CSV",49.59,0.0


Full per-run detail, including cleaned-vs-raw 

In [6]:
strategy_eval[["source", "Run", "File", "Semantic Precision", "Semantic Recall", "Semantic F1",
               "Profession Acc"]]

,source,Run,File,Semantic Precision,Semantic Recall,Semantic F1,Profession Acc
0,llm_reading/run_1/raw,llm_reading_run_1_raw,1947_gc_page_70_gpt_semantic_extraction.csv,54.92,54.47,54.69,0.00
1,llm_reading/run_1/cleaned,llm_reading_run_1_cleaned,1947_gc_page_70_gpt_semantic_extraction.csv,91.80,91.06,91.43,98.21
2,llm_reading_to_semantic/run_1/raw,llm_reading_to_semantic_run_1_raw,straight_reading_to_semantic.csv,88.62,88.62,88.62,100.00
3,llm_reading_to_semantic/run_1/cleaned,llm_reading_to_semantic_run_1_cleaned,straight_reading_to_semantic.csv,91.80,91.06,91.43,100.00
4,ocr_llm_to_semantic/run_1/raw,ocr_llm_to_semantic_run_1_raw,straight_reading_to_semantic.csv,49.59,49.59,49.59,0.00


In [ ]:
# Pair every raw file row with its cleaned counterpart.

_pairs = []
for _, row in strategy_eval.iterrows():
    run = row["Run"]
    if not isinstance(run, str) or not run.endswith("_raw") or pd.isna(row["Semantic F1"]):
        continue
    base = run[:-len("_raw")]
    cleaned = strategy_eval[
        (strategy_eval["Run"] == base + "_cleaned") & strategy_eval["Semantic F1"].notna()
    ]
    if cleaned.empty:
        continue
    c = cleaned.iloc[0]
    _pairs.append(dict(
        run=base,
        semantic_f1_raw=row["Semantic F1"], semantic_f1_cleaned=c["Semantic F1"],
        profession_acc_raw=row["Profession Acc"], profession_acc_cleaned=c["Profession Acc"],
        address_acc_raw=row["Address Acc"], address_acc_cleaned=c["Address Acc"],
        name_acc_raw=row["Name Acc"], name_acc_cleaned=c["Name Acc"],
    ))

raw_vs_cleaned = pd.DataFrame(_pairs)
raw_vs_cleaned

,run,semantic_f1_raw,semantic_f1_cleaned,profession_acc_raw,profession_acc_cleaned,address_acc_raw,address_acc_cleaned,name_acc_raw,name_acc_cleaned
0,llm_reading_run_1,54.69,91.43,0.0,98.21,100.00,94.64,100.00,66.96
1,llm_reading_to_semantic_run_1,88.62,91.43,100.0,100.00,88.07,88.39,73.39,72.32


## Choice 3 — Final pipeline: GPT-4o, full-LLM, no reasoning model (13-page exploration, 22/04)

In [8]:
# Structural output per page

rows = []
for csv_path in sorted(EXPLORATION_OUTPUTS.glob("page_*.csv"), key=lambda p: int(p.stem.split('_')[1])):
    page = int(csv_path.stem.split("_")[1])
    txt_path = csv_path.with_suffix(".txt")
    with open(csv_path, encoding="utf-8", errors="replace") as f:
        parsed_rows = list(csv.reader(f.read().splitlines()))
    rows.append(dict(
        page=page,
        csv_rows=max(len(parsed_rows) - 1, 0),
        columns=", ".join(parsed_rows[0]) if parsed_rows else "",
        txt_lines=len(txt_path.read_text(encoding="utf-8", errors="replace").splitlines()) if txt_path.exists() else None,
    ))

exploration = pd.DataFrame(rows)
exploration

,page,csv_rows,columns,txt_lines
0,4,0,add,22
1,10,115,"Name, Address, Profession, Additional Info",33
2,14,26,"Name, Page Number",124
3,36,147,"Name, Title, Diocese, Additional Info",106
4,42,48,"Name, Position, Department, Additional Info",106
5,45,80,"Name, Address, Profession, Additional Info",77
6,49,60,"Name, Title, Additional Info",107
7,58,0,"Name, Address, Profession, Additional Info",1
8,60,396,"Name, Address, Role, Affiliation, Additional Info",109
9,63,368,"Name, Title, Role, Address, Phone, Organization",116


In [9]:
# Two pages produced zero rows -- check why.
for page in exploration.loc[exploration.csv_rows == 0, "page"]:
    txt = (EXPLORATION_OUTPUTS / f"page_{page}.txt").read_text(encoding="utf-8")
    print(f"--- page {page}.txt ---")
    print(txt[:200])
    print()

--- page 4.txt ---
DITTA CAV. G. PELLEGNINI  
- VENEZIA -  
CAMPO S. BARTOLOMEO N. 5379 - TEL. 25-004  

Addizionatrici e calcolatrici  
a mano ed elettriche  
Macchine per scrivere  
Affrancatrici postali "Francotyp"  

--- page 58.txt ---
I'm sorry, I can't do that.



In [10]:
GROUND_TRUTH = Path(GROUND_TRUTH_DIR)
exploration_eval = pd.read_csv(CHOICE3_DIR / "exploration_evaluation_results.csv")

unreadable = []
for p in sorted(GROUND_TRUTH.glob("**/*")):
    if not p.is_file():
        continue
    try:
        p.read_bytes()
    except OSError:
        unreadable.append(p.name)
if unreadable:
    print(f"WARNING -- unreadable/corrupted ground-truth files: {unreadable}")

TWELVE_PAGES = [4, 10, 14, 36, 42, 45, 58, 63, 87, 103, 151, 167]

ground_truth_pages = sorted(
    int(p.stem.replace("page_", "").replace("_ground_truth", ""))
    for p in GROUND_TRUTH.glob("txt/*_ground_truth.txt")
)
print(f"Ground truth available for: {ground_truth_pages}")
print(f"Twelve-page diversity set: {TWELVE_PAGES}")

exploration_eval["page_num"] = exploration_eval["Page"].str.replace("page_", "").astype(int)
exploration_eval[
    (exploration_eval["Type"] == "Semantic") & exploration_eval["page_num"].isin(TWELVE_PAGES)
][["Page", "Run", "Semantic Precision", "Semantic Recall", "Semantic F1", "Row Count Similarity"]]

Ground truth available for: [10, 14, 36, 42, 45, 58, 60, 63, 86, 87, 103, 110, 151, 167]
Twelve-page diversity set: [4, 10, 14, 36, 42, 45, 58, 63, 87, 103, 151, 167]


,Page,Run,Semantic Precision,Semantic Recall,Semantic F1,Row Count Similarity
1,page_4,page_4_semantic_1,0.00,0.00,0.00,50.00
3,page_10,page_10_semantic_1,90.32,90.32,90.32,100.00
5,page_14,page_14_semantic_1,98.04,98.04,98.04,100.00
7,page_36,page_36_semantic_1,96.67,96.67,96.67,100.00
10,page_42,page_42_semantic_1,92.86,100.00,96.30,92.86
11,page_42,page_42_semantic_2,50.00,53.85,51.85,92.86
15,page_45,page_45_semantic_1,0.00,0.00,0.00,97.62
16,page_45,page_45_semantic_2,100.00,100.00,100.00,100.00
20,page_58,page_58_semantic_1,71.43,67.57,69.44,94.59
21,page_58,page_58_semantic_2,100.00,100.00,100.00,100.00


In [11]:
# Run-to-run similarity for pages extracted more than once 
exploration_eval[exploration_eval["Type"] == "Run Comparison"][["Page", "Run-to-Run Similarity"]]

,Page,Run-to-Run Similarity
12,page_42,0.00
17,page_45,26.19
22,page_58,64.86
39,page_151,86.36
